# Visual Multi-Task ResNet-50 Pipeline

Predicting **both** Exact Bilirubin (Regression) AND Treatment Risk (Classification) simultaneously.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight

plt.rcParams['figure.figsize'] = (15, 5)

# Paths
CSV_PATH = '/Users/shubhbhateja/College/minor_project/NeoJaundice/chd_jaundice_published_2.csv'
IMAGE_DIR = '/Users/shubhbhateja/College/minor_project/images'
INPUT_SIZE = (128, 128)

## 1. Load Data
Let's see what the CSV looks like. We are predicting both `Treatment` and `blood(mg/dL)`.

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Total rows in CSV: {len(df)}")
display(df.head())

# Check class balance
print("\nClass Distribution (Treatment):")
print(df['Treatment'].value_counts(normalize=True) * 100)

print("\nBilirubin Stats:")
print(df['blood(mg/dL)'].describe())

## 2. Step-by-Step Visualization of Preprocessing
Plotting every step so we can see if the HSV thresholding (Skin/Yellow Patch detection) and White Balancing are working properly.

In [ ]:
def demonstrate_preprocessing(image_name):
    img_path = os.path.join(IMAGE_DIR, image_name)
    raw_bgr = cv2.imread(img_path)
    if raw_bgr is None:
        print(f"Could not load {img_path}")
        return
        
    raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
    h, w = raw_rgb.shape[:2]
    
    # STEP 1: White Balance (Gray World)
    result = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2LAB)
    avg_a = np.average(result[:, :, 1])
    avg_b = np.average(result[:, :, 2])
    result[:, :, 1] = result[:, :, 1] - ((avg_a - 128) * (result[:, :, 0] / 255.0) * 1.1)
    result[:, :, 2] = result[:, :, 2] - ((avg_b - 128) * (result[:, :, 0] / 255.0) * 1.1)
    balanced_bgr = cv2.cvtColor(result, cv2.COLOR_LAB2BGR)
    balanced_rgb = cv2.cvtColor(balanced_bgr, cv2.COLOR_BGR2RGB)
    
    # STEP 2: Skin Detection via YCrCb (Adaptive Crop Logic)
    img_ycrcb = cv2.cvtColor(balanced_rgb, cv2.COLOR_RGB2YCrCb)
    mask = cv2.inRange(img_ycrcb, np.array([0, 133, 77]), np.array([255, 173, 127]))
    mask[int(h*0.85):h, :] = 0 
    
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    cleaned_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    
    contours, _ = cv2.findContours(cleaned_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        x, y, w_b, h_b = cv2.boundingRect(max(contours, key=cv2.contourArea))
        ix, iy = int(w_b*0.05), int(h_b*0.05)
        skin_patch = balanced_rgb[y+iy:y+h_b-iy, x+ix:x+w_b-ix]
    else:
        skin_patch = balanced_rgb
        
    # STEP 3: Final Tensors for ResNet
    resized_rgb = cv2.resize(skin_patch, INPUT_SIZE)
    hsv_tensor = cv2.cvtColor(resized_rgb, cv2.COLOR_RGB2HSV)
    
    # PLOT EVERYTHING
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    axes[0].imshow(raw_rgb)
    axes[0].set_title("1. Raw Image (RGB)")
    
    axes[1].imshow(balanced_rgb)
    axes[1].set_title("2. White Balanced (Gray World)")
    
    axes[2].imshow(cleaned_mask, cmap='gray')
    axes[2].set_title("3. Skin Threshold Mask")
    
    axes[3].imshow(resized_rgb)
    axes[3].set_title(f"4. Final RGB Crop {INPUT_SIZE}")
    
    axes[4].imshow(hsv_tensor)
    axes[4].set_title(f"5. Final HSV Tensor {INPUT_SIZE}")
    
    plt.tight_layout()
    plt.show()

    return resized_rgb / 255.0, hsv_tensor / 255.0

sample_image = df.iloc[50]['image_idx']
print(f"Visualizing pipeline for: {sample_image}")
rgb_norm, hsv_norm = demonstrate_preprocessing(sample_image)

## 3. Create Multi-Target Dataset Tensors & Train/Val/Test Split
Now we extract BOTH the `Treatment` label AND the `blood(mg/dL)` value for Multi-Task Learning. We perform a strict **70-10-20 Split** utilizing `train_test_split` twice.

In [ ]:
# Exact production preprocessing logic without plotting for speed
def prepare_image_2d(img_path):
    img = cv2.imread(img_path)
    if img is None: return None, None
    
    result = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    avg_a = np.average(result[:, :, 1])
    avg_b = np.average(result[:, :, 2])
    result[:, :, 1] = result[:, :, 1] - ((avg_a - 128) * (result[:, :, 0] / 255.0) * 1.1)
    result[:, :, 2] = result[:, :, 2] - ((avg_b - 128) * (result[:, :, 0] / 255.0) * 1.1)
    balanced = cv2.cvtColor(result, cv2.COLOR_LAB2BGR)
    rgb = cv2.cvtColor(balanced, cv2.COLOR_BGR2RGB)
    
    h, w = rgb.shape[:2]
    img_ycrcb = cv2.cvtColor(rgb, cv2.COLOR_RGB2YCrCb)
    mask = cv2.inRange(img_ycrcb, np.array([0, 133, 77]), np.array([255, 173, 127]))
    mask[int(h*0.85):h, :] = 0 
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        skin_rgb = cv2.resize(rgb, INPUT_SIZE)
    else:
        x, y, w_b, h_b = cv2.boundingRect(max(contours, key=cv2.contourArea))
        ix, iy = int(w_b*0.05), int(h_b*0.05)
        skin_rgb = rgb[y+iy:y+h_b-iy, x+ix:x+w_b-ix]
        
    resized_rgb = cv2.resize(skin_rgb, INPUT_SIZE)
    hsv = cv2.cvtColor(resized_rgb, cv2.COLOR_RGB2HSV)
    
    return resized_rgb / 255.0, hsv / 255.0

# --- LOAD FULL DATASET --- #
print("Be patient, processing all images... this may take a couple of minutes.")
X_rgb, X_hsv, Y_treat, Y_bili = [], [], [], []

for idx, row in df.iterrows():
    path = os.path.join(IMAGE_DIR, row['image_idx'])
    rgb_img, hsv_img = prepare_image_2d(path)
    if rgb_img is not None:
        X_rgb.append(rgb_img)
        X_hsv.append(hsv_img)
        Y_treat.append(row['Treatment'])  # Task 1: Classification
        Y_bili.append(row['blood(mg/dL)']) # Task 2: Regression
    if idx % 500 == 0:
        print(f"Processed {idx} images...")

X_rgb = np.array(X_rgb)
X_hsv = np.array(X_hsv)
Y_treat = np.array(Y_treat)
Y_bili = np.array(Y_bili)
print(f"Final Dataset Size: {len(X_rgb)} samples")

In [ ]:
# --- 70% / 10% / 20% SPLIT --- #
# 1. Split out Test set first (20%)
# We stratify by Treatment to ensure consistent class balance in all splits
X_rgb_temp, X_rgb_ts, X_hsv_temp, X_hsv_ts, yt_temp, yt_ts, yb_temp, yb_ts = train_test_split(
    X_rgb, X_hsv, Y_treat, Y_bili, test_size=0.20, random_state=42, stratify=Y_treat
)

# 2. Split remaining 80% into Train (70%) and Val (10%)
# So test_size for Val = 10 / 80 = 0.125
X_rgb_tr, X_rgb_val, X_hsv_tr, X_hsv_val, yt_tr, yt_val, yb_tr, yb_val = train_test_split(
    X_rgb_temp, X_hsv_temp, yt_temp, yb_temp, test_size=0.125, random_state=42, stratify=yt_temp
)

print(f"Training Set: {len(X_rgb_tr)} samples (70%)")
print(f"Validation Set: {len(X_rgb_val)} samples (10%)")
print(f"Testing Set: {len(X_rgb_ts)} samples (20%)")

# Compute class weights for imbalanced classification training
weights = class_weight.compute_class_weight('balanced', classes=np.unique(yt_tr), y=yt_tr)
dict_weights = {0: weights[0], 1: weights[1]}
print(f"Class Weights (Treatment): {dict_weights}")

## 4. Build Multi-Task ResNet-50 Model (Fine-Tuning)
We define TWO output heads: `treatment_output` (sigmoid) and `bilirubin_output` (linear).
We also UNFREEZE the top layers of ResNet-50 so the model doesn't "guess" randomly anymore!

In [ ]:
def build_mtl_resnet50_model():
    # Inputs
    input_rgb = layers.Input(shape=(INPUT_SIZE[0], INPUT_SIZE[1], 3), name='rgb_input')
    input_hsv = layers.Input(shape=(INPUT_SIZE[0], INPUT_SIZE[1], 3), name='hsv_input')

    # Shared Backbone
    shared_resnet = applications.ResNet50(include_top=False, weights='imagenet', input_shape=(INPUT_SIZE[0], INPUT_SIZE[1], 3))
    
    # UNFREEZE THE BACKBONE (Fine-tuning for better feature extraction!)
    # We freeze earlier layers, unfreeze the last few blocks.
    shared_resnet.trainable = True
    for layer in shared_resnet.layers[:-30]:
        layer.trainable = False

    # Process RGB
    x_rgb = shared_resnet(input_rgb)
    x_rgb = layers.GlobalAveragePooling2D()(x_rgb)

    # Process HSV
    x_hsv = shared_resnet(input_hsv)
    x_hsv = layers.GlobalAveragePooling2D()(x_hsv)

    # Fusion
    merged = layers.Concatenate()([x_rgb, x_hsv])
    
    # HEAD 1: TREATMENT (Classification)
    x_class = layers.Dense(256, activation='relu')(merged)
    x_class = layers.Dropout(0.4)(x_class)
    x_class = layers.Dense(128, activation='relu')(x_class)
    out_treatment = layers.Dense(1, activation='sigmoid', name='treatment_output')(x_class)

    # HEAD 2: BILIRUBIN (Regression)
    x_reg = layers.Dense(256, activation='relu')(merged)
    x_reg = layers.Dropout(0.2)(x_reg)
    x_reg = layers.Dense(128, activation='relu')(x_reg)
    out_bilirubin = layers.Dense(1, activation='linear', name='bilirubin_output')(x_reg)

    # KEY FIX: Return lists, mapping by index (Index 0 = Treatment, Index 1 = Bilirubin)
    model = models.Model(
        inputs=[input_rgb, input_hsv], 
        outputs=[out_treatment, out_bilirubin]
    )
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), # Smaller LR for fine-tuning
        loss=['binary_crossentropy', 'mse'],
        loss_weights=[10.0, 1.0], # Index 0 gets weight 10, Index 1 gets weight 1
        metrics=[['accuracy'], ['mae']]
    )
    
    return model

model = build_mtl_resnet50_model()
print("Multi-Task Architecture built.")

## 5. Training Loop
We track training history for both outputs.

In [ ]:
def lr_schedule(epoch):
    lr = 1e-4
    if epoch > 15: lr *= 0.5
    return lr

# Early Stopping on Total Validation Loss
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=10, 
    restore_best_weights=True
)

# Create explicit sample weight lists
sw_treat = np.array([dict_weights[int(label)] for label in yt_tr])
sw_bili = np.ones(len(yb_tr))

print("Starting Multi-Task ResNet-50 Fine-Tuning...")
history = model.fit(
    x=[X_rgb_tr, X_hsv_tr], # Same order as inputs=[input_rgb, input_hsv]
    y=[yt_tr, yb_tr], # Same order as outputs=[out_treatment, out_bilirubin]
    validation_data=( 
        [X_rgb_val, X_hsv_val], 
        [yt_val, yb_val]
    ),
    sample_weight=[sw_treat, sw_bili],
    epochs=40,
    batch_size=16,
    callbacks=[
        tf.keras.callbacks.LearningRateScheduler(lr_schedule),
        early_stopping
    ],
    verbose=1
)

model.save('mtl_resnet50_jaundice_model.h5')
print("Model saved as mtl_resnet50_jaundice_model.h5")

## 6. Evaluate Performance on Unseen Test Set

In [ ]:
# 1. Evaluate Model using Lists
test_scores = model.evaluate(
    [X_rgb_ts, X_hsv_ts], 
    [yt_ts, yb_ts]
)
print(f"\nFINAL UNSEEN TEST EVALUATION:")
print(f"Total Loss: {test_scores[0]:.4f}")
# Keras evaluate returns: [total_loss, loss1, loss2, metric1, metric2]
# In Keras 3 with lists: scores are highly dependent on the metrics listed.
# Usually: [loss, treatment_loss, bilirubin_loss, treatment_accuracy, bilirubin_mae]
print(f"Treatment Accuracy: {test_scores[3]*100:.2f}%")
print(f"Bilirubin MAE: {test_scores[4]:.2f} mg/dL")

# 2. Plot Training Curves
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Total Loss
axes[0].plot(history.history['loss'], label='Train Total Loss')
axes[0].plot(history.history['val_loss'], label='Val Total Loss')
axes[0].set_title('Total Model Loss')
axes[0].legend()

# Classification Accuracy
axes[1].plot(history.history['treatment_output_accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_treatment_output_accuracy'], label='Val Accuracy')
axes[1].set_title('Treatment Classification Accuracy')
axes[1].legend()

# Regression Error (MAE)
axes[2].plot(history.history['bilirubin_output_mae'], label='Train Bilirubin MAE')
axes[2].plot(history.history['val_bilirubin_output_mae'], label='Val Bilirubin MAE')
axes[2].set_title('Bilirubin Regression Error (MAE)')
axes[2].legend()

plt.tight_layout()
plt.show()